# Notebook 4: Feature Engineering
**Project:** Car Price Estimation  
**Goal:** Create new meaningful features, extract information, and transform skewed variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('../data/02_cleaned_data.csv')
print(f'Loaded: {df.shape}')
df.head(3)

## 1. Extract Brand & Model from CarName

In [ ]:
# Brand already extracted in cleaning; extract model (rest of CarName)
df['model'] = df['CarName'].apply(lambda x: ' '.join(str(x).split()[1:]) if len(str(x).split()) > 1 else 'unknown')
print('Sample brand/model:')
print(df[['CarName', 'brand', 'model']].head(10))

## 2. Brand Segment (Luxury / Mid-range / Economy)

In [ ]:
luxury_brands   = ['jaguar', 'buick', 'porsche', 'volvo', 'mercedes-benz', 'bmw', 'alfa-romeo']
midrange_brands = ['audi', 'saab', 'volkswagen', 'honda', 'nissan', 'mitsubishi',
                   'mazda', 'subaru', 'isuzu', 'mercury', 'peugeot', 'renault', 'dodge']

def classify_brand(brand):
    if brand in luxury_brands:   return 'luxury'
    if brand in midrange_brands: return 'mid_range'
    return 'economy'

df['brand_segment'] = df['brand'].apply(classify_brand)
print('Brand segment distribution:')
print(df['brand_segment'].value_counts())

## 3. Power-to-Weight Ratio

In [ ]:
df['power_to_weight'] = df['horsepower'] / df['curbweight']
print('Power-to-weight ratio stats:')
print(df['power_to_weight'].describe())

## 4. Engine Size Category

In [ ]:
def categorize_engine(size):
    if size <= 100:   return 'small'
    if size <= 160:   return 'medium'
    return 'large'

df['engine_size_cat'] = df['enginesize'].apply(categorize_engine)
print('Engine size category distribution:')
print(df['engine_size_cat'].value_counts())

## 5. MPG Average & Fuel Efficiency Score

In [ ]:
df['avg_mpg'] = (df['citympg'] + df['highwaympg']) / 2

def fuel_efficiency(mpg):
    if mpg >= 35:  return 'high'
    if mpg >= 25:  return 'medium'
    return 'low'

df['fuel_efficiency_cat'] = df['avg_mpg'].apply(fuel_efficiency)
print('Fuel efficiency distribution:')
print(df['fuel_efficiency_cat'].value_counts())

## 6. Car Size Score

In [ ]:
# Normalize and combine length, width, height into size score
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
size_matrix = scaler.fit_transform(df[['carlength', 'carwidth', 'carheight']])
df['car_size_score'] = size_matrix.mean(axis=1)
print('Car size score stats:')
print(df['car_size_score'].describe())

## 7. Log Transformation for Skewed Features

In [ ]:
skewed_cols = ['price', 'enginesize', 'horsepower', 'curbweight']

fig, axes = plt.subplots(len(skewed_cols), 2, figsize=(14, 4 * len(skewed_cols)))

for i, col in enumerate(skewed_cols):
    # Original
    sns.histplot(df[col], kde=True, ax=axes[i][0], color='steelblue')
    axes[i][0].set_title(f'{col} - Original (skew={df[col].skew():.2f})')

    # Log transformed
    log_col = np.log1p(df[col])
    df[f'log_{col}'] = log_col
    sns.histplot(log_col, kde=True, ax=axes[i][1], color='coral')
    axes[i][1].set_title(f'log_{col} - Log Transformed (skew={log_col.skew():.2f})')

plt.tight_layout()
plt.savefig('../reports/04_log_transformations.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Binning Continuous Variables

In [ ]:
# Bin horsepower into categories
df['hp_category'] = pd.cut(df['horsepower'],
                            bins=[0, 80, 120, 160, 300],
                            labels=['low', 'medium', 'high', 'very_high'])

# Bin price into segments
df['price_segment'] = pd.cut(df['price'],
                              bins=[0, 10000, 20000, 30000, 100000],
                              labels=['budget', 'mid', 'premium', 'luxury'])

print('Horsepower category:')
print(df['hp_category'].value_counts())
print('\nPrice segment:')
print(df['price_segment'].value_counts())

## 9. Feature Summary & Save

In [ ]:
new_features = ['brand', 'model', 'brand_segment', 'power_to_weight',
                'engine_size_cat', 'avg_mpg', 'fuel_efficiency_cat',
                'car_size_score', 'log_price', 'log_enginesize',
                'log_horsepower', 'log_curbweight', 'hp_category', 'price_segment']

print(f'New features created: {len(new_features)}')
print(f'Total features: {df.shape[1]}')
df[new_features].head()


In [ ]:
df.to_csv('../data/04_engineered_data.csv', index=False)
print('Engineered dataset saved to data/04_engineered_data.csv')
print('Notebook 4 Complete!')